In [1]:
import warnings
import mne
import random
import numpy as np
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.signal import stft

sampling_rate = 256
duration      = 2.8
clip_length   = int(sampling_rate * duration)   # 716 samples
window_size = 10 # Length of window (samples)
step_size = 5 # Length of step (samples)

patients = [
    {
        "id": "patient_01",
        "normal_edf":    "chb01/chb01_01.edf",
        "pre_ictal_edf": "chb01/chb01_16.edf",
        "seizure_start_sec": 1015,   # patient-specific
    },
    {
        "id": "patient_02",
        "normal_edf":    "chb02/chb02_01.edf",
        "pre_ictal_edf": "chb02/chb02_16+.edf",
        "seizure_start_sec": 2972,
    },
    {
        "id": "patient_03",
        "normal_edf":    "chb03/chb03_06.edf",
        "pre_ictal_edf": "chb03/chb03_02.edf",
        "seizure_start_sec": 731,
    },
    {
        "id": "patient_04",
        "normal_edf":    "chb05/chb05_01.edf",
        "pre_ictal_edf": "chb05/chb05_13.edf",
        "seizure_start_sec": 1086,
    }
]

def build_patient_data(normal_windows, pre_ictal_windows):
    """Takes windowed data, applies STFT, returns X and y_labels."""
    from scipy.signal import stft
    
    # ── Combine windows ───────────────────────────────────────────
    all_normal = np.vstack(list(normal_windows.values()))
    X_windows  = np.vstack([all_normal, pre_ictal_windows])
    y_labels = np.hstack([
        np.zeros(len(all_normal)),
        np.ones(len(pre_ictal_windows))
    ]).astype(int)        

    # ── STFT each window ──────────────────────────────────────────
    stft_features = []
    for w in X_windows:
        _, _, Zxx = stft(w, nperseg=10)
        stft_features.append(np.log10(np.abs(Zxx) ** 2 + 1e-10))
    stft_features = np.array(stft_features)

    # ── Flatten ───────────────────────────────────────────────────
    X = stft_features.reshape(stft_features.shape[0], -1)
    
    return X, y_labels


def train_patient_model(X, y_labels):
    """Splits, scales, trains MLP, returns report."""
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_labels, test_size=0.2, random_state=42, stratify=y_labels
    )

    print(f"  y_train: {dict(zip(*np.unique(y_train, return_counts=True)))}")
    print(f"  y_test:  {dict(zip(*np.unique(y_test, return_counts=True)))}")
    print(f"  X shape: {X.shape}")

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    mlp = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        max_iter=2000,
        early_stopping=True,
        random_state=42
    )
    # compute weights based on class frequency
    from sklearn.utils.class_weight import compute_sample_weight
    sample_weights = compute_sample_weight(class_weight={0: 1, 1: 5}, y=y_train)
    
    mlp.fit(X_train, y_train, sample_weight=sample_weights)

    y_pred = mlp.predict(X_test)
    report = classification_report(y_test, y_pred, digits=4, output_dict=True)

    print(f"  Train accuracy : {mlp.score(X_train, y_train):.4f}")
    print(f"  Test accuracy  : {mlp.score(X_test, y_test):.4f}")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, digits=4))

    return report

def moving_window(data, window_size, step_size):
    num_points = len(data)
    window_length = window_size # Length of window in samples
    step_length = step_size # Length of step in samples
    windows = []
    for start in range(0, num_points - window_length + 1, step_length):
        windows.append(data[start:start + window_length])
    return np.array(windows)

all_reports = []

for patient in patients:
    print(f"\n{'='*40}")
    print(f"Patient: {patient['id']}")
    print(f"{'='*40}")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        raw       = mne.io.read_raw_edf(patient["normal_edf"],    preload=True, verbose=False)
        pre_ictal = mne.io.read_raw_edf(patient["pre_ictal_edf"], preload=True, verbose=False)

    # ── Extract raw clips (your existing code) ────────────────────
    raw          = mne.io.read_raw_edf(patient["normal_edf"],    preload=True, verbose=False)
    pre_ictal    = mne.io.read_raw_edf(patient["pre_ictal_edf"], preload=True, verbose=False)

    # Normal clips
    total_samples = raw.n_times
    normal_clips_list = []
    for _ in range(10):
        start_sample = random.randint(0, total_samples - clip_length)
        chunk        = raw.get_data(start=start_sample, stop=start_sample + clip_length)
        normal_clips_list.append(chunk)
    X_normal = np.array(normal_clips_list)

    # Pre-ictal clip
    seizure_start    = patient["seizure_start_sec"]
    sph              = seizure_start - 600
    start_pre_ictal  = int(sph * sampling_rate)
    pre_ictal_data   = pre_ictal.get_data()
    pre_ictal_clip   = pre_ictal_data[0, start_pre_ictal:start_pre_ictal + clip_length]

    # ── Window (your existing code) ───────────────────────────────
    normal_windows = {}
    for i in range(10):
        normal_windows[f"clip_{i}"] = moving_window(X_normal[i, 0, :], window_size, step_size)
    pre_ictal_windows = moving_window(pre_ictal_clip, window_size, step_size)

    # ── Build dataset + train ─────────────────────────────────────
    X, y_labels  = build_patient_data(normal_windows, pre_ictal_windows)
    report       = train_patient_model(X, y_labels)
    all_reports.append({"id": patient["id"], "report": report})

# ── Mean across all patients ──────────────────────────────────────
print(f"\n{'='*40}")
print("MEAN ACROSS ALL PATIENTS")
print(f"{'='*40}")
for label, name in [("0", "Normal"), ("1", "Pre-ictal")]:
    avg_p = np.mean([r["report"][label]["precision"] for r in all_reports])
    avg_r = np.mean([r["report"][label]["recall"]    for r in all_reports])
    avg_f = np.mean([r["report"][label]["f1-score"]  for r in all_reports])
    print(f"  {name:10s} → precision: {avg_p:.4f}  recall: {avg_r:.4f}  f1: {avg_f:.4f}")
avg_acc = np.mean([r["report"]["accuracy"] for r in all_reports])
print(f"  Accuracy   → {avg_acc:.4f}")


Patient: patient_01


C:\Users\Mary Yap\AppData\Local\Temp\ipykernel_15900\3452761851.py:131: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw          = mne.io.read_raw_edf(patient["normal_edf"],    preload=True, verbose=False)
C:\Users\Mary Yap\AppData\Local\Temp\ipykernel_15900\3452761851.py:132: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  pre_ictal    = mne.io.read_raw_edf(patient["pre_ictal_edf"], preload=True, verbose=False)


  y_train: {np.int64(0): np.int64(1135), np.int64(1): np.int64(114)}
  y_test:  {np.int64(0): np.int64(285), np.int64(1): np.int64(28)}
  X shape: (1562, 18)
  Train accuracy : 0.8927
  Test accuracy  : 0.8850
[[261  24]
 [ 12  16]]
              precision    recall  f1-score   support

           0     0.9560    0.9158    0.9355       285
           1     0.4000    0.5714    0.4706        28

    accuracy                         0.8850       313
   macro avg     0.6780    0.7436    0.7030       313
weighted avg     0.9063    0.8850    0.8939       313


Patient: patient_02


C:\Users\Mary Yap\AppData\Local\Temp\ipykernel_15900\3452761851.py:131: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw          = mne.io.read_raw_edf(patient["normal_edf"],    preload=True, verbose=False)
C:\Users\Mary Yap\AppData\Local\Temp\ipykernel_15900\3452761851.py:132: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  pre_ictal    = mne.io.read_raw_edf(patient["pre_ictal_edf"], preload=True, verbose=False)


  y_train: {np.int64(0): np.int64(1135), np.int64(1): np.int64(114)}
  y_test:  {np.int64(0): np.int64(285), np.int64(1): np.int64(28)}
  X shape: (1562, 18)
  Train accuracy : 0.9656
  Test accuracy  : 0.9776
[[278   7]
 [  0  28]]
              precision    recall  f1-score   support

           0     1.0000    0.9754    0.9876       285
           1     0.8000    1.0000    0.8889        28

    accuracy                         0.9776       313
   macro avg     0.9000    0.9877    0.9382       313
weighted avg     0.9821    0.9776    0.9787       313


Patient: patient_03


C:\Users\Mary Yap\AppData\Local\Temp\ipykernel_15900\3452761851.py:131: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw          = mne.io.read_raw_edf(patient["normal_edf"],    preload=True, verbose=False)
C:\Users\Mary Yap\AppData\Local\Temp\ipykernel_15900\3452761851.py:132: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  pre_ictal    = mne.io.read_raw_edf(patient["pre_ictal_edf"], preload=True, verbose=False)


  y_train: {np.int64(0): np.int64(1135), np.int64(1): np.int64(114)}
  y_test:  {np.int64(0): np.int64(285), np.int64(1): np.int64(28)}
  X shape: (1562, 18)
  Train accuracy : 0.9143
  Test accuracy  : 0.9042
[[281   4]
 [ 26   2]]
              precision    recall  f1-score   support

           0     0.9153    0.9860    0.9493       285
           1     0.3333    0.0714    0.1176        28

    accuracy                         0.9042       313
   macro avg     0.6243    0.5287    0.5335       313
weighted avg     0.8632    0.9042    0.8749       313


Patient: patient_04


C:\Users\Mary Yap\AppData\Local\Temp\ipykernel_15900\3452761851.py:131: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw          = mne.io.read_raw_edf(patient["normal_edf"],    preload=True, verbose=False)
C:\Users\Mary Yap\AppData\Local\Temp\ipykernel_15900\3452761851.py:132: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  pre_ictal    = mne.io.read_raw_edf(patient["pre_ictal_edf"], preload=True, verbose=False)


  y_train: {np.int64(0): np.int64(1135), np.int64(1): np.int64(114)}
  y_test:  {np.int64(0): np.int64(285), np.int64(1): np.int64(28)}
  X shape: (1562, 18)
  Train accuracy : 0.9039
  Test accuracy  : 0.9010
[[277   8]
 [ 23   5]]
              precision    recall  f1-score   support

           0     0.9233    0.9719    0.9470       285
           1     0.3846    0.1786    0.2439        28

    accuracy                         0.9010       313
   macro avg     0.6540    0.5753    0.5955       313
weighted avg     0.8751    0.9010    0.8841       313


MEAN ACROSS ALL PATIENTS
  Normal     → precision: 0.9487  recall: 0.9623  f1: 0.9548
  Pre-ictal  → precision: 0.4795  recall: 0.4554  f1: 0.4303
  Accuracy   → 0.9169


In [11]:
print(f"  Class counts: {dict(zip(*np.unique(y_labels, return_counts=True)))}")

  Class counts: {np.int64(0): np.int64(40), np.int64(1): np.int64(4)}


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from scipy.signal import stft as scipy_stft

window_size = 256
step_size   = 5

# ── SET-SVD helpers ───────────────────────────────────────────────

def train_patient_model_set_svd(X, y_labels, patient_id, method_name):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_labels, test_size=0.2, random_state=42, stratify=y_labels
    )

    print(f"  y_train: {dict(zip(*np.unique(y_train, return_counts=True)))}")
    print(f"  y_test:  {dict(zip(*np.unique(y_test,  return_counts=True)))}")
    print(f"  X shape: {X.shape}")

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    mlp = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        max_iter=2000,
        early_stopping=True,
        random_state=42
    )

    sample_weights = compute_sample_weight(class_weight={0: 5, 1: 1}, y=y_train)
    mlp.fit(X_train, y_train, sample_weight=sample_weights)

    y_pred = mlp.predict(X_test)
    report = classification_report(y_test, y_pred, digits=4, output_dict=True)

    print(f"  Train accuracy : {mlp.score(X_train, y_train):.4f}")
    print(f"  Test accuracy  : {mlp.score(X_test,  y_test):.4f}")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, digits=4))

    return report

def synchroextracting_transform(signal, fs=256, nperseg=10, noverlap=5, eps=1e-6):
    hop = nperseg - noverlap
    dt  = hop / fs

    f, t, X = scipy_stft(signal, fs=fs, nperseg=nperseg, noverlap=noverlap,
                          boundary=None, padded=False)
    df = f[1] - f[0] if len(f) > 1 else 1.0

    dX_dt = np.zeros_like(X)
    if X.shape[1] > 1:
        dX_dt[:, :-1] = (X[:, 1:] - X[:, :-1]) / dt
        dX_dt[:,  -1] = dX_dt[:, -2]

    denom = np.where(np.abs(X) < eps, eps, X)
    IF    = f[:, None] + np.imag(dX_dt / (2.0 * np.pi * denom))

    mask = np.abs(IF - f[:, None]) <= (df / 2.0)
    SET  = np.where(mask, X, 0.0 + 0.0j)

    return np.abs(SET)

def set_svd_features(signal, fs=256, nperseg=10, noverlap=5, n_sv=10):
    """
    SET matrix → SVD → top-n_sv singular values.
    """
    SET_matrix = synchroextracting_transform(signal, fs=fs,
                                             nperseg=nperseg, noverlap=noverlap)
    _, sigma, _ = np.linalg.svd(SET_matrix, full_matrices=False)

    k        = min(n_sv, len(sigma))
    features = sigma[:k]
    if k < n_sv:                          # pad if matrix is tiny
        features = np.pad(features, (0, n_sv - k))
    return features   # shape: (10,)

def build_patient_data_set_svd(normal_windows, pre_ictal_windows):
    all_normal = np.vstack(list(normal_windows.values()))
    X_windows  = np.vstack([all_normal, pre_ictal_windows])
    y_labels   = np.hstack([
        np.zeros(len(all_normal)),
        np.ones(len(pre_ictal_windows))
    ]).astype(int)

    features = []
    for w in X_windows:
        SET_matrix = synchroextracting_transform(w, fs=256, nperseg=10, noverlap=5)
        _, sigma, _ = np.linalg.svd(SET_matrix, full_matrices=False)
        k    = min(10, len(sigma))
        feat = np.pad(sigma[:k], (0, 10 - k))
        features.append(feat)

    X = np.array(features)
    return X, y_labels

    # ── Main loop ─────────────────────────────────────────────────────

all_reports_set_svd = []

for patient in patients:
    print(f"\n{'='*40}")
    print(f"Patient: {patient['id']}  [SET-SVD MLP]")
    print(f"{'='*40}")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        raw       = mne.io.read_raw_edf(patient["normal_edf"],    preload=True, verbose=False)
        pre_ictal = mne.io.read_raw_edf(patient["pre_ictal_edf"], preload=True, verbose=False)

    # Normal clips
    total_samples    = raw.n_times
    normal_clips_list = []
    for _ in range(720):
        start_sample = random.randint(0, total_samples - clip_length)
        chunk        = raw.get_data(start=start_sample, stop=start_sample + clip_length)
        normal_clips_list.append(chunk)
    X_normal = np.array(normal_clips_list)

    # Pre-ictal clip
    seizure_start   = patient["seizure_start_sec"]
    sph             = seizure_start - 600
    start_pre_ictal = int(sph * sampling_rate)
    pre_ictal_data  = pre_ictal.get_data()
    pre_ictal_clip  = pre_ictal_data[0, start_pre_ictal:start_pre_ictal + clip_length]

    # Windows
    normal_windows = {}
    for i in range(720):
        normal_windows[f"clip_{i}"] = moving_window(X_normal[i, 0, :], window_size, step_size)
    pre_ictal_windows = moving_window(pre_ictal_clip, window_size, step_size)

    # Build + train
    X, y_labels = build_patient_data_set_svd(normal_windows, pre_ictal_windows)
    report      = train_patient_model_set_svd(X, y_labels, patient["id"], "SET-SVD")
    all_reports_set_svd.append({"id": patient["id"], "report": report})

# ── Mean across all patients ──────────────────────────────────────
print(f"\n{'='*40}")
print("MEAN ACROSS ALL PATIENTS  [SET-SVD MLP]")
print(f"{'='*40}")
for label, name in [("0", "Normal"), ("1", "Pre-ictal")]:
    avg_p = np.mean([r["report"][label]["precision"] for r in all_reports_set_svd])
    avg_r = np.mean([r["report"][label]["recall"]    for r in all_reports_set_svd])
    avg_f = np.mean([r["report"][label]["f1-score"]  for r in all_reports_set_svd])
    print(f"  {name:10s} → precision: {avg_p:.4f}  recall: {avg_r:.4f}  f1: {avg_f:.4f}")
avg_acc = np.mean([r["report"]["accuracy"] for r in all_reports_set_svd])
print(f"  Accuracy   → {avg_acc:.4f}")

In [2]:
from ssqueezepy import ssq_cwt, extract_ridges
from ssqueezepy.experimental import scale_to_freq

def compute_SET(clip, sr=256):
    """
    Synchroextracting Transform of a 1D EEG clip.
    Returns time-frequency matrix (freq_bins, time_frames).
    """
    Tx, _, ssq_freqs, *_ = synsq_cwt(clip, fs=sr)
    return np.abs(Tx)              # (freq_bins, time_frames)

def compute_SET_SVD(clip, sr=256, n_components=10):
    """
    Full SET-SVD pipeline for one clip/window.
    Returns 1D feature vector of singular values, shape (n_components,).
    """
    TF_matrix = compute_SET(clip, sr=sr)         # (freq_bins, time_frames)
    
    U, S, Vt = np.linalg.svd(TF_matrix, full_matrices=False)
    
    return S[:n_components]                       # keep top-k singular values

In [3]:
def build_patient_data_setsvd(normal_windows, pre_ictal_windows, n_svd=10):
    all_normal = np.vstack(list(normal_windows.values()))
    X_windows  = np.vstack([all_normal, pre_ictal_windows])
    y_labels   = np.hstack([
        np.zeros(len(all_normal)),
        np.ones(len(pre_ictal_windows))
    ]).astype(int)

    features = []
    for w in X_windows:
        feat = compute_SET_SVD(w, n_components=n_svd)   # (n_svd,)
        features.append(feat)

    X = np.array(features)     # (n_windows, n_svd)
    return X, y_labels

for patient in patients:
    print(f"\n{'='*40}")
    print(f"Patient: {patient['id']}")
    print(f"{'='*40}")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        raw       = mne.io.read_raw_edf(patient["normal_edf"],    preload=True, verbose=False)
        pre_ictal = mne.io.read_raw_edf(patient["pre_ictal_edf"], preload=True, verbose=False)

    # ── Extract raw clips (your existing code) ────────────────────
    raw          = mne.io.read_raw_edf(patient["normal_edf"],    preload=True, verbose=False)
    pre_ictal    = mne.io.read_raw_edf(patient["pre_ictal_edf"], preload=True, verbose=False)

    # Normal clips
    total_samples = raw.n_times
    normal_clips_list = []
    for _ in range(10):
        start_sample = random.randint(0, total_samples - clip_length)
        chunk        = raw.get_data(start=start_sample, stop=start_sample + clip_length)
        normal_clips_list.append(chunk)
    X_normal = np.array(normal_clips_list)

    # Pre-ictal clip
    seizure_start    = patient["seizure_start_sec"]
    sph              = seizure_start - 600
    start_pre_ictal  = int(sph * sampling_rate)
    pre_ictal_data   = pre_ictal.get_data()
    pre_ictal_clip   = pre_ictal_data[0, start_pre_ictal:start_pre_ictal + clip_length]

    # ── Window (your existing code) ───────────────────────────────
    normal_windows = {}
    for i in range(10):
        normal_windows[f"clip_{i}"] = moving_window(X_normal[i, 0, :], window_size, step_size)
    pre_ictal_windows = moving_window(pre_ictal_clip, window_size, step_size)

    # ── Build dataset + train ─────────────────────────────────────
    X, y_labels  = build_patient_data_setsvd(normal_windows, pre_ictal_windows)
    report       = train_patient_model(X, y_labels)
    all_reports.append({"id": patient["id"], "report": report})


Patient: patient_01


C:\Users\Mary Yap\AppData\Local\Temp\ipykernel_15900\3162558545.py:28: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw          = mne.io.read_raw_edf(patient["normal_edf"],    preload=True, verbose=False)
C:\Users\Mary Yap\AppData\Local\Temp\ipykernel_15900\3162558545.py:29: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  pre_ictal    = mne.io.read_raw_edf(patient["pre_ictal_edf"], preload=True, verbose=False)


NameError: name 'synsq_cwt' is not defined

In [10]:
print(f"  Class counts: {dict(zip(*np.unique(y_labels, return_counts=True)))}")

  Class counts: {np.int64(0): np.int64(40), np.int64(1): np.int64(4)}
